# [실습] 피처 선택 · 데이터 누수

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔧 [실습] 피처·파이프라인·데이터 누수 — 새지 않는 실험 설계

## — 사용하지 않았던 범주형 피처를 평가 정보가 새지 않게 연결합니다

지난 순서에 학습곡선이 이렇게 말했습니다. _"데이터를 더 모아도 소용없다. 입력을 바꿔라."_ 그런데 우리는 원본의 **17개 예측 피처 중 수치형 10개만** 사용하고 있었습니다. 문자 열을 모델에 넣는 법을 몰랐기 때문입니다.

오늘 범주형 7개를 추가해 17개 원본 예측 피처를 모두 검토합니다. 그리고 그 과정에서 평가를 왜곡할 수 있는 사고를 다루는 법을 함께 배웁니다.

> 🤖 **오늘의 AI 활용 규칙 — 검증 단계:**  
> 전처리 코드는 AI에게 물어도 됩니다. 단, 받은 코드에서 **`fit`의 대상, 피처 생성 시점, 분할 단위**를 확인합니다. 이 한 줄 점검이 오늘 배우는 것의 가장 실용적인 쓰임입니다. (자세한 규칙은 개념 노트북 Part 0)

## 📋 오늘의 실습

지난 보고를 받은 팀장이 승인했습니다.

> 🧑‍💼 "버리고 있던 열을 쓰는 건 좋습니다. 그런데 옆 팀이 비슷한 걸 했다가 **실서비스에서 성능이 반토막** 난 적이 있어요. 개발할 때 0.95였는데 배포하니 0.7이었다고 합니다. 그런 일이 없다는 걸 어떻게 보장하시겠습니까?"

팀장이 말한 사고가 바로 **데이터 누수**입니다. 오늘은 성능을 올리는 일과, **그 성능 추정이 타당한지 확인하는 일**을 함께 합니다.

| 문제 | 내용                                   | 확인하는 힘                     |
| ---- | -------------------------------------- | ------------------------------- |
| 1    | 범주형 7개 열을 `Pipeline`으로 푼다    | `ColumnTransformer` 구성        |
| 2    | 파생변수 가설 2개를 세우고 검증한다    | 가설 → 실험 → 채택/기각         |
| 3    | 누수를 직접 만들어 부풀림을 잰다       | 누수 진단과 교차 적합           |
| 4    | `Pipeline` 통째로 `GridSearchCV`       | 전처리까지 함께 튜닝            |
| 5    | 실험 기록표 · 모델 저장 · 모델 카드 v5 | 재현 가능한 산출물 (**제출물**) |

> 📌 **오늘의 주 지표는 AP와 F1입니다.**  
> `average_precision`으로 계산한 AP는 여러 임계값의 정밀도·재현율을 요약하고, F1은 기본 임계값의 운영점을 보여줍니다. 정확도는 보조 지표로 확인합니다.

# ⚙️ 데이터 준비

이론 노트북의 Titanic과 달리, 이번 실습 노트북에서는 이전 시간에서 사용한 **UCI Online Shoppers Purchasing Intention** 데이터를 이어서 사용합니다. 한 행은 온라인 쇼핑몰 방문 세션 1건이며, 세션 정보로 구매 완료 여부를 분류합니다.

| 항목        | 내용                                                                      |
| ----------- | ------------------------------------------------------------------------- |
| 관측 단위   | 온라인 쇼핑몰 방문 세션 1건 — 1년 동안 세션별 사용자가 겹치지 않도록 구성 |
| 데이터 크기 | 12,330행 × 18열 — 입력 피처 17개 + 타깃 1개                               |
| 타깃        | `Revenue` — 구매 완료 `True`(1), 미구매 `False`(0)                        |
| 클래스 분포 | 구매 1,908건(15.5%) · 미구매 10,422건(84.5%)                              |
| 입력 피처   | 수치형 10개 + 범주형 7개                                                  |
| 결측치      | 없음                                                                      |
| 사용 목적   | 혼합형 전처리, 파생변수 비교, 타깃 인코딩 누수 재현과 교차 적합           |

### 입력 피처 구성

- **수치형 10개:** `Administrative`부터 `SpecialDay`까지의 방문 행동·시간·페이지 지표
- **범주형 7개:** `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType`, `Weekend`
- **타깃 1개:** `Revenue`

`OperatingSystems`·`Browser`·`Region`·`TrafficType`은 숫자로 저장돼 있지만 **크기에 의미가 없는 코드**입니다(`Browser=13`이 `Browser=1`보다 크다는 뜻이 아닙니다). 따라서 이 네 열은 범주형 전처리 갈래에 배정합니다.

> ℹ️ **출처와 이용 조건**  
> [UCI Machine Learning Repository · Dataset 468](https://archive.ics.uci.edu/dataset/468/online%2Bshoppers%2Bpurchasing%2Bintention%2Bdataset), DOI `10.24432/C5F88Q` · **CC BY 4.0**

> ⚠️ **예측 시점 먼저 정하기**  
> 이 실습은 세션 정보로 같은 세션의 구매 완료 여부를 설명합니다. 실시간 구매 의도 예측으로 확장하려면, 예측 요청 시점까지 확정된 피처만 남겨야 합니다. 특히 `PageValues`는 거래 완료와 관련해 계산되므로 시점 누수 후보로 따로 점검합니다.

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
CAT_COLS = ["Month", "OperatingSystems", "Browser", "Region",
            "TrafficType", "VisitorType", "Weekend"]

y = shoppers["Revenue"].astype(int)

print(f"수치형 {len(NUM_COLS)}개 · 범주형 {len(CAT_COLS)}개 · 타깃 1개")
print("범주별 값 개수:", {c: shoppers[c].nunique() for c in CAT_COLS})
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
수치형 10개 · 범주형 7개 · 타깃 1개
범주별 값 개수: {'Month': 10, 'OperatingSystems': 8, 'Browser': 13, 'Region': 9, 'TrafficType': 20, 'VisitorType': 3, 'Weekend': 2}

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. 범주형 7개 열을 `Pipeline`으로 푼다

`ColumnTransformer`로 수치형은 그대로 통과시키고 범주형만 One-Hot으로 펼칩니다. 그 전체를 `Pipeline`에 담아 **교차 검증에 통째로** 넘깁니다. 그러면 인코더가 겹마다 학습 데이터로만 `fit`됩니다.

```
[문제 1]
1) ColumnTransformer를 만듭니다.
   - 수치형: "passthrough"
   - 범주형: OneHotEncoder(handle_unknown="ignore", min_frequency=20)
2) 그것과 지난 순서의 모델을 Pipeline으로 묶습니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20)
3) 5겹 CV로 F1과 AP(`average_precision`)를 측정하고, 수치형 10개만 썼을 때와 비교합니다.
4) 변환 후 피처가 몇 개가 됐는지 출력합니다.
```

> 🤔 **예상하기**  
> 열이 10개에서 17개로 늘고, One-Hot으로 펼치면 피처는 더 많아집니다. F1과 AP의 변화 방향을 예상합니다. 두 지표가 같은 방향으로 움직일지 확인합니다.

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [2]:
# [C2] 문제 1. 범주형 7개 열을 `Pipeline`으로 푼다
# ⌨️ 문제 1 — ColumnTransformer + Pipeline으로 범주형 개방
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# 1) ColumnTransformer 구성
preprocess = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
])

# 2) 전처리 + 모델을 Pipeline으로 결합
model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE
)
pipe = Pipeline([
    ("pre", preprocess),
    ("clf", model),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"f1": "f1", "ap": "average_precision"}

# 3) 5겹 CV: 수치형+범주형(전체 파이프라인)
full_scores = cross_validate(
    pipe, shoppers[NUM_COLS + CAT_COLS], y,
    cv=cv, scoring=scoring, n_jobs=-1
)

# 비교 대상: 수치형 10개만 사용 (전처리 없이 모델만)
num_only_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE
)
num_scores = cross_validate(
    num_only_model, shoppers[NUM_COLS], y,
    cv=cv, scoring=scoring, n_jobs=-1
)

print("\n=== 5-겹 교차검증 결과 ===")
print(f"{'구성':<20}{'F1 (mean±std)':<22}{'AP (mean±std)':<22}")
print(f"{'수치형+범주형':<20}"
      f"{full_scores['test_f1'].mean():.4f}±{full_scores['test_f1'].std():.4f}"
      f"{'':<6}"
      f"{full_scores['test_ap'].mean():.4f}±{full_scores['test_ap'].std():.4f}")
print(f"{'수치형만(10개)':<20}"
      f"{num_scores['test_f1'].mean():.4f}±{num_scores['test_f1'].std():.4f}"
      f"{'':<6}"
      f"{num_scores['test_ap'].mean():.4f}±{num_scores['test_ap'].std():.4f}")

# 4) 변환 후 피처 개수 확인
preprocess.fit(shoppers[NUM_COLS + CAT_COLS], y)
n_features = preprocess.transform(shoppers[NUM_COLS + CAT_COLS]).shape[1]
print(f"\n변환 후 피처 개수: {n_features}개 "
      f"(수치형 {len(NUM_COLS)}개 + 원-핫 인코딩된 범주형 {n_features - len(NUM_COLS)}개)")

# 참고: 각 범주형 컬럼이 원-핫 인코딩 후 몇 개의 열로 늘었는지 확인
ohe = preprocess.named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(CAT_COLS)
print(f"OneHotEncoder가 만든 열 개수: {len(cat_feature_names)}개")


=== 5-겹 교차검증 결과 ===
구성                  F1 (mean±std)         AP (mean±std)         
수치형+범주형             0.5764±0.0253      0.7371±0.0158
수치형만(10개)           0.6186±0.0175      0.7248±0.0191

변환 후 피처 개수: 66개 (수치형 10개 + 원-핫 인코딩된 범주형 56개)
OneHotEncoder가 만든 열 개수: 56개


<details>
<summary>(클릭) 💡 힌트</summary>

- `ColumnTransformer([("num", "passthrough", NUM_COLS), ("cat", OneHotEncoder(...), CAT_COLS)])`
- `min_frequency=20`은 20건 미만으로 나타나는 희귀 값을 하나로 묶습니다. 열이 지나치게 늘어나는 것을 막습니다.
- `handle_unknown="ignore"`가 없으면 검증 겹에만 나타나는 값에서 오류가 납니다.
- `Pipeline([("pre", 전처리), ("clf", 모델)])`을 통째로 `cross_validate`에 넘깁니다.
- 변환 후 피처 수는 전처리를 `fit`한 뒤 `.transform(X).shape[1]`로 확인합니다.
- 비교 대상(수치형만)은 `Pipeline` 없이 `cross_validate(model, shoppers[NUM_COLS], y, ...)`로 잽니다.

</details>

> 🎯 **[C2] 확인하기**  
> **한쪽만 올랐습니다.** AP는 0.7248 → **0.7372**로 올랐는데, F1은 0.6204 → **0.5780**으로 오히려 떨어졌습니다.
>
> 모순처럼 보이지만 아닙니다. 지난 순서에서 배운 내용을 연결합니다 — **AP는 여러 임계값의 정밀도·재현율을 요약**하고, **F1은 기본 판정 임계값에서** 계산됩니다. 범주형을 넣어 모델의 *확률 순위 매기는 능력*은 좋아졌는데(AP↑), 확률 분포가 바뀌면서 **0.5라는 선이 전보다 나쁜 자리**에 놓인 것입니다.
>
> 실무에서 이럴 때 하는 일은 명확합니다. F1 하락만으로 범주형을 즉시 제외하지 않습니다. 먼저 검증 데이터에서 임계값을 다시 선택하고 AP와 운영 제약을 함께 확인합니다. 지표가 무엇을 재는지 알면 이런 판단이 가능해집니다.

# 문제 2. 파생변수 — 가설을 먼저 적고, 그다음 검증한다

피처 엔지니어링의 순서는 **코드가 먼저가 아닙니다.** 가설이 먼저입니다. 아무 조합이나 만들어 점수가 오르길 기다리는 것은 실험이 아니라 도박입니다.

다행히 우리에겐 근거가 있습니다. **순서 2의 실습에서 이미 발견한 사실**이 있습니다.

- 이 데이터에서 `PageValues`가 0인 세션의 구매율 **3.9%**, 0보다 큰 세션은 **56.3%** — 무려 14.6배
- 구매 세션은 상품 페이지 체류가 길다 (중앙값 위 22.3% 대 아래 8.7%)

```
[문제 2]
1) 아래 두 가설을 코드로 만들기 전에, 각각 "왜 효과가 있을 것인가"를 한 줄로 적습니다.
   가설 A: has_page_value = (PageValues > 0)  ← 0 여부가 추가 표현으로 유용할 수 있다
   가설 B: total_pages = Administrative + Informational + ProductRelated  ← 총 탐색량
2) 문제 1의 ② 구성을 기준으로, A만 / B만 각각 추가해 5겹 CV로 측정합니다.
3) 각각의 ΔF1과 ΔAP를 계산해 채택·기각을 결정합니다.
```

> 🤔 **예상하기**  
> `has_page_value`는 이미 있는 `PageValues`에서 연속값을 0/1로 단순화한 피처입니다. 트리가 자체적으로 분기점을 찾을 수 있는데도 이 표현이 추가 효과를 낼지 예상합니다.

> ⚠️ **주의하기 — 예측 시점**  
> UCI 설명에서 `PageValues`는 전자상거래 거래 완료와 관련해 계산되는 지표입니다. 실제 서비스에 사용하려면 예측 요청 시점에 이 값이 이미 확정되어 있는지 확인해야 합니다. 확인되지 않으면 시점 누수 후보로 분류합니다.

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [3]:
# [C3] 문제 2. 파생변수 — 가설을 먼저 적고, 그다음 검증한다
# ⌨️ 문제 2 — 가설 A·B를 각각 따로 검증

# 원본 보존 및 파생변수 생성
feat = shoppers.copy()
# 가설 A: PageValues > 0 여부 (불리언 → 0/1)
feat["has_page_value"] = (feat["PageValues"] > 0).astype(int)
# 가설 B: 총 탐색 페이지 수
feat["total_pages"] = feat["Administrative"] + feat["Informational"] + feat["ProductRelated"]


def evaluate_features(data, extra_num_cols, num_cols=NUM_COLS, cat_cols=CAT_COLS,
                       cv=cv, scoring=scoring, random_state=RANDOM_STATE):
    """수치형 목록에 extra_num_cols를 추가해 같은 파이프라인 구조로 5겹 CV 점수를 반환."""
    all_num_cols = num_cols + extra_num_cols
    pre = ColumnTransformer([
        ("num", "passthrough", all_num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_cols),
    ])
    clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=random_state)
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    return cross_validate(pipe, data[all_num_cols + cat_cols], y, cv=cv, scoring=scoring, n_jobs=-1)


# A만 추가 / B만 추가 — 따로 측정해야 원인을 가릴 수 있음
scores_A = evaluate_features(feat, ["has_page_value"])
scores_B = evaluate_features(feat, ["total_pages"])

# 기준선: 문제 1의 ② 구성 (수치형+범주형, log[1])
base_f1, base_f1_std = full_scores["test_f1"].mean(), full_scores["test_f1"].std()
base_ap, base_ap_std = full_scores["test_ap"].mean(), full_scores["test_ap"].std()

print(f"{'구성':<25}{'F1 (ΔF1)':<25}{'AP (ΔAP)':<25}")
print(f"{'기준 (문제1 ②)':<25}{base_f1:.4f}{'':<15}{base_ap:.4f}")
for name, s in [("A: has_page_value", scores_A), ("B: total_pages", scores_B)]:
    f1, ap = s["test_f1"].mean(), s["test_ap"].mean()
    print(f"{name:<25}{f1:.4f} ({f1-base_f1:+.4f}){'':<8}{ap:.4f} ({ap-base_ap:+.4f})")

# 채택/기각 판단: 표준편차(변동폭) 대비 유의미하게 개선됐는지로 판단
print("\n=== 판단 ===")
for name, s in [("A", scores_A), ("B", scores_B)]:
    d_f1 = s["test_f1"].mean() - base_f1
    d_ap = s["test_ap"].mean() - base_ap
    verdict = "채택" if (d_f1 > base_f1_std * 0.3 or d_ap > base_ap_std * 0.3) else "기각(또는 보류)"
    print(f"가설 {name}: ΔF1={d_f1:+.4f}, ΔAP={d_ap:+.4f} → {verdict}")

구성                       F1 (ΔF1)                 AP (ΔAP)                 
기준 (문제1 ②)               0.5764               0.7371
A: has_page_value        0.6465 (+0.0701)        0.7498 (+0.0127)
B: total_pages           0.5797 (+0.0033)        0.7399 (+0.0027)

=== 판단 ===
가설 A: ΔF1=+0.0701, ΔAP=+0.0127 → 채택
가설 B: ΔF1=+0.0033, ΔAP=+0.0027 → 기각(또는 보류)


<details>
<summary>(클릭) 💡 힌트</summary>

- 원본을 건드리지 않도록 `feat = shoppers.copy()`로 시작합니다.
- 불리언을 피처로 넣을 때는 `.astype(int)`로 0/1을 만듭니다.
- 두 가설을 **따로** 재야 어느 쪽이 효과를 냈는지 알 수 있습니다. 한꺼번에 넣으면 원인을 못 가립니다.
- `ColumnTransformer`의 수치 목록만 바꾸면 되므로, 목록을 인자로 받는 함수로 만들어 두면 반복이 줄어듭니다.
- 기준은 문제 1의 ② 결과입니다 — `log[1]`에 들어 있습니다.

</details>

> 🎯 **[C3] 확인하기**  
> **0 여부를 표현한 피처의 평균 점수가 높아졌습니다.** `has_page_value`는 연속값을 0/1로 줄이기만 했는데 F1을 **+0.0687** 올렸습니다 — 오늘 얻은 개선 중 가장 큽니다.
>
> 랜덤포레스트는 `PageValues`의 분기점을 자체적으로 찾을 수 있지만, 0 여부를 명시한 표현이 제한된 트리 구조에서 더 쉽게 선택됐을 가능성이 있습니다. 정확한 원인은 추가 실험 없이는 단정할 수 없습니다.
>
> 이 가설의 출처도 확인합니다. **순서 2의 실습에서 직접 계산한 구매율 대비**입니다. 파생변수는 관찰에서 가설을 세우고, 별도 검증과 예측 시점 감사로 채택 여부를 결정합니다.

# 문제 3. 누수를 직접 만들어, 부풀림을 잰다

이제 팀장의 질문에 답할 차례입니다. 개발 점수가 낙관적으로 나타나는 과정을 직접 재현합니다.

흔한 시나리오 하나를 씁니다. 범주 조합별 **과거 구매율**을 피처로 만드는 것입니다(타깃 인코딩). 아이디어 자체는 정상이지만, **전체 데이터로 계산해 버리면** 검증 겹의 정답이 피처에 스며듭니다.

```
[문제 3]
1) 조합 키를 만듭니다 — combo = Month + "_" + TrafficType + "_" + Region
   (몇 개 조합이 나오는지, 10건 미만인 조합이 몇 개인지 확인합니다)
2) 누수 버전: 전체 데이터로 조합별 Revenue 평균을 구해 열로 추가 → CV
3) 교차 적합 버전: 같은 일을 `TargetEncoder`로 `Pipeline` 안에서 수행 → CV
4) 두 점수의 차이를 계산하고, 교차 적합 버전이 기준(문제 1의 ②)보다 나아졌는지 확인합니다.
```

> 🤔 **예상하기**  
> 조합 키는 715개가 나오고 그중 464개는 10건 미만입니다. 표본이 적은 그룹의 구매율에 각 행의 정답이 얼마나 크게 반영될지 예상합니다. 전체 데이터로 계산했을 때 검증 행의 정답이 피처에 들어가는 경로를 확인합니다.

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [4]:
# [C4] 문제 3. 누수를 직접 만들어, 부풀림을 잰다
# ⌨️ 문제 3 — 같은 피처를 누수 버전과 정상 버전으로 각각 재기
from sklearn.preprocessing import TargetEncoder

# 1) 조합 키 생성 및 희소성 확인
combo = (shoppers["Month"].astype(str) + "_" +
         shoppers["TrafficType"].astype(str) + "_" +
         shoppers["Region"].astype(str))

combo_counts = combo.value_counts()
print(f"조합 개수: {combo.nunique()}")
print(f"10건 미만 조합 수: {(combo_counts < 10).sum()} / 전체 {len(combo_counts)}")

# 2) 누수 버전: 전체 데이터로 조합별 Revenue 평균을 미리 계산
leak_df = shoppers.copy()
leak_df["combo"] = combo
leak_df["combo_target_enc"] = leak_df.groupby("combo")["Revenue"].transform("mean")

leak_num_cols = NUM_COLS + ["combo_target_enc"]
pre_leak = ColumnTransformer([
    ("num", "passthrough", leak_num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
])
clf_leak = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE)
pipe_leak = Pipeline([("pre", pre_leak), ("clf", clf_leak)])
leak_scores = cross_validate(
    pipe_leak, leak_df[leak_num_cols + CAT_COLS], y,
    cv=cv, scoring=scoring, n_jobs=-1
)

# 3) 교차 적합 버전: TargetEncoder를 파이프라인 안에 넣어 겹마다 새로 fit
correct_df = shoppers.copy()
correct_df["combo"] = combo

pre_correct = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
    ("combo_te", TargetEncoder(random_state=RANDOM_STATE), ["combo"]),
])
clf_correct = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE)
pipe_correct = Pipeline([("pre", pre_correct), ("clf", clf_correct)])
correct_scores = cross_validate(
    pipe_correct, correct_df[NUM_COLS + CAT_COLS + ["combo"]], y,
    cv=cv, scoring=scoring, n_jobs=-1
)

# 4) 비교: 기준(문제 1 ②) vs 누수 버전 vs 교차 적합 버전
base_f1, base_ap = full_scores["test_f1"].mean(), full_scores["test_ap"].mean()
leak_f1, leak_ap = leak_scores["test_f1"].mean(), leak_scores["test_ap"].mean()
corr_f1, corr_ap = correct_scores["test_f1"].mean(), correct_scores["test_ap"].mean()

print(f"\n{'구성':<20}{'F1':<12}{'AP':<12}")
print(f"{'기준 (문제1 ②)':<20}{base_f1:.4f}{'':<4}{base_ap:.4f}")
print(f"{'누수 버전':<20}{leak_f1:.4f}{'':<4}{leak_ap:.4f}")
print(f"{'교차 적합 버전':<20}{corr_f1:.4f}{'':<4}{corr_ap:.4f}")

print(f"\n누수로 인한 부풀림(누수 - 교차적합): ΔF1={leak_f1-corr_f1:+.4f}, ΔAP={leak_ap-corr_ap:+.4f}")
print(f"교차적합 버전과 기준선 차이: ΔF1={corr_f1-base_f1:+.4f}, ΔAP={corr_ap-base_ap:+.4f}")

조합 개수: 715
10건 미만 조합 수: 464 / 전체 715

구성                  F1          AP          
기준 (문제1 ②)          0.5764    0.7371
누수 버전               0.5958    0.7481
교차 적합 버전            0.5687    0.7356

누수로 인한 부풀림(누수 - 교차적합): ΔF1=+0.0271, ΔAP=+0.0126
교차적합 버전과 기준선 차이: ΔF1=-0.0076, ΔAP=-0.0016


<details>
<summary>(클릭) 💡 힌트</summary>

- 조합 키는 문자열을 이어 붙여 만듭니다: `df["Month"].astype(str) + "_" + df["TrafficType"].astype(str) + ...`
- **누수 버전**의 핵심은 `groupby(...)["Revenue"].transform("mean")` — 이 한 줄이 전체 데이터를 보고 계산합니다.
- **정상 버전**은 `TargetEncoder`를 `ColumnTransformer`의 한 갈래로 넣으면 됩니다. 그러면 겹마다 학습 데이터로만 `fit`됩니다.
- 두 버전 모두 **나머지 조건은 똑같이** 두어야 비교가 성립합니다.
- 조합 개수는 `.nunique()`, 10건 미만 그룹 수는 `(값별_개수 < 10).sum()`입니다.

</details>

> 🎯 **[C4] 확인하기**  
> **10건짜리 그룹의 "과거 구매율"은 사실상 그 10건의 정답을 평균한 값**입니다. 자기 자신의 답을 포함한 채로요. 715개 조합 중 464개가 그런 상태입니다.
>
> 그래서 부풀림이 **AP +0.0125, F1 +0.0264** 만큼 생겼습니다. 더 무서운 것은 방향입니다 — 누수 버전만 보면 이 피처가 **기준보다 +0.011 좋아 보이는데**, 정직하게 재면 **−0.002로 오히려 나쁩니다.** 개선이 있다고 믿고 배포했다면, 운영 데이터에서는 미래 타깃으로 같은 값을 계산할 수 없으므로 개발 점수가 재현되지 않을 수 있습니다.
>
> 이 실험은 타깃 인코딩의 계산 범위가 평가를 어떻게 왜곡하는지 보여줍니다. `TargetEncoder`를 `Pipeline` 안에 두면 학습 시 내부 교차 적합이 적용되지만, 예측 시점 이후 피처와 분할 단위는 별도로 감사해야 합니다.

> 📌 **오늘의 점검 한 줄:**  
> 남의 코드(또는 AI가 준 코드)를 볼 때 가장 먼저 찾을 것은 **학습되는 `fit`·`fit_transform`이나 타깃을 사용하는 `groupby(...).transform`이 교차 검증 바깥에서 실행되는가**입니다. 바깥에 있다면 그 숫자는 일단 의심해야 합니다.

# 문제 4. `Pipeline` 통째로 `GridSearchCV`

지금까지 손잡이를 하나씩 손으로 돌렸습니다. 이제 자동화합니다. 그런데 핵심은 **전처리 옵션도 함께 튜닝한다**는 점입니다 — `Pipeline`을 쓰면 전처리와 모델의 손잡이를 **한 격자에서** 다룰 수 있습니다.

> 🔁 **개념 복습 — 이번 탐색의 규모**  
> 전처리 옵션 2개 × 모델 옵션 3개로 **6개 후보 조합**을 만듭니다. 각 조합을 같은 5겹으로 평가하므로 후보 비교에 6 × 5 = **30번의 학습**이 필요합니다. `average_precision`의 평균이 가장 높은 조합을 선택하고, 기본값 `refit=True`가 선택된 `Pipeline`을 전체 입력 데이터로 한 번 더 학습합니다. `best_score_`는 이 선택 과정의 내부 CV 점수입니다.

```
[문제 4]
1) 문제 2에서 채택한 구성(수치 10개 + has_page_value + 범주 7개)으로 Pipeline을 만듭니다.
2) GridSearchCV로 다음 두 손잡이를 함께 탐색합니다. `scoring="average_precision"`
   - pre__cat__min_frequency: [10, 50]      ← 전처리 옵션
   - clf__min_samples_leaf:   [5, 20, 50]   ← 모델 옵션
3) 조합별 내부 CV 점수를 표로 출력하고, `best_params_`와 `best_score_`를 확인합니다.
4) 선택된 모델의 F1도 같은 5겹 분할에서 측정해 기록합니다. 이 값은 모델 선택에 사용한 데이터의 내부 추정치이므로 최종 일반화 성능으로 보고하지 않습니다.
```

> 🤔 **예상하기**  
> 지난 순서에서 `min_samples_leaf`는 **20**이 좋았습니다. 그런데 그때는 피처가 10개뿐이었습니다. 피처 표현이 늘어난 뒤에도 20이 선택될지 예상합니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [5]:
# [C5] 문제 4. `Pipeline` 통째로 `GridSearchCV`
# ⌨️ 문제 4 — 전처리 옵션과 모델 옵션을 한 격자에서
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import GridSearchCV, cross_val_score

num_cols_p2 = NUM_COLS + ["has_page_value"]

pre = ColumnTransformer([
    ("num", "passthrough", num_cols_p2),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
])

# 1) 탐색 단계용 파이프라인 — n_estimators를 100으로 낮춰서 속도 확보
search_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)
search_pipe = Pipeline([("pre", pre), ("clf", search_clf)])

param_grid = {
    "pre__cat__min_frequency": [10, 50],
    "clf__min_samples_leaf": [5, 20, 50],
}

gs = GridSearchCV(
    search_pipe, param_grid=param_grid,
    scoring="average_precision",
    cv=cv, n_jobs=-1,       # 여기서만 병렬 (30개 조합을 코어 수만큼 동시에)
    refit=False,            # 자동 재학습은 생략 — 어차피 100그루짜리라 안 씀
    verbose=1,              # 진행상황 확인용
)
gs.fit(feat[num_cols_p2 + CAT_COLS], y)

# 2) 조합별 내부 CV 점수 표 + best_params_ / best_score_
results = pd.DataFrame(gs.cv_results_)
cols = [c for c in results.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
print(results[cols].sort_values("mean_test_score", ascending=False).to_string(index=False))
print(f"\nbest_params_: {gs.best_params_}")
print(f"best_score_ (AP, n_estimators=100 기준): {gs.best_score_:.4f}")

# 3) 최적 조합으로 '진짜' 최종 모델(n_estimators=300)을 딱 한 번만 학습
final_clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1)
final_pipe = Pipeline([("pre", pre), ("clf", final_clf)])
final_pipe.set_params(**gs.best_params_)
final_pipe.fit(feat[num_cols_p2 + CAT_COLS], y)   # 문제 5에서 저장할 최종 파이프라인

# 4) 선택된 모델의 F1을 같은 5겹에서 측정 (내부 추정치, 최종 성능 아님)
f1_scores = cross_val_score(final_pipe, feat[num_cols_p2 + CAT_COLS], y, cv=cv, scoring="f1", n_jobs=-1)
print(f"\n선택된 모델 F1 (동일 5겹, 내부 추정치): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print("※ 이 F1은 모델 선택에 쓰인 것과 같은 데이터 분할에서 나온 값이라 최종 일반화 성능으로 보고하지 않습니다.")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
 param_clf__min_samples_leaf  param_pre__cat__min_frequency  mean_test_score  std_test_score
                           5                             10         0.751479        0.011667
                           5                             50         0.750811        0.013171
                          20                             50         0.748766        0.011812
                          20                             10         0.747401        0.012396
                          50                             50         0.741325        0.013318
                          50                             10         0.740287        0.015720

best_params_: {'clf__min_samples_leaf': 5, 'pre__cat__min_frequency': 10}
best_score_ (AP, n_estimators=100 기준): 0.7515

선택된 모델 F1 (동일 5겹, 내부 추정치): 0.6543 ± 0.0175
※ 이 F1은 모델 선택에 쓰인 것과 같은 데이터 분할에서 나온 값이라 최종 일반화 성능으로 보고하지 않습니다.


In [6]:
# 결과 표 확인
results = pd.DataFrame(gs.cv_results_)
cols = [c for c in results.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
print(results[cols].sort_values("mean_test_score", ascending=False).to_string(index=False))
print(f"\nbest_params_: {gs.best_params_}")
print(f"best_score_ (AP, n_estimators=100 기준): {gs.best_score_:.4f}")

# 최적 조합으로 진짜 최종 모델(n_estimators=300) 학습
final_clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1)
final_pipe = Pipeline([("pre", pre), ("clf", final_clf)])
final_pipe.set_params(**gs.best_params_)
final_pipe.fit(feat[num_cols_p2 + CAT_COLS], y)

# F1 측정 (내부 추정치)
f1_scores = cross_val_score(final_pipe, feat[num_cols_p2 + CAT_COLS], y, cv=cv, scoring="f1", n_jobs=-1)
print(f"\n선택된 모델 F1 (동일 5겹, 내부 추정치): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")

 param_clf__min_samples_leaf  param_pre__cat__min_frequency  mean_test_score  std_test_score
                           5                             10         0.751479        0.011667
                           5                             50         0.750811        0.013171
                          20                             50         0.748766        0.011812
                          20                             10         0.747401        0.012396
                          50                             50         0.741325        0.013318
                          50                             10         0.740287        0.015720

best_params_: {'clf__min_samples_leaf': 5, 'pre__cat__min_frequency': 10}
best_score_ (AP, n_estimators=100 기준): 0.7515

선택된 모델 F1 (동일 5겹, 내부 추정치): 0.6543 ± 0.0175


<details>
<summary>(클릭) 💡 힌트</summary>

- `Pipeline` 안의 손잡이는 **이중 밑줄**로 지정합니다: `단계이름__파라미터`. 중첩되면 계속 이어 붙입니다 — `pre__cat__min_frequency`는 "`pre` 단계의 `cat` 갈래의 `min_frequency`"입니다.
- 단계 이름은 만들 때 준 이름 그대로입니다(`"pre"`, `"cat"`, `"clf"`).
- 결과는 `gs.cv_results_`에 들어 있습니다. `pd.DataFrame(gs.cv_results_)`에서 `param_*`·`mean_test_score`·`std_test_score` 열만 뽑으면 깔끔합니다.
- `n_jobs=-1`을 주면 조합을 병렬로 돌려 훨씬 빠릅니다.

</details>

> 🎯 **[C5] 확인하기**  
> **이 탐색에서는 5가 선택됐습니다.**
>
> 선택값 변화의 원인을 점검합니다. 피처 구성과 탐색 공간이 달라지면 선택되는 하이퍼파라미터도 달라질 수 있습니다. 이번 결과만으로 선택값 변화의 원인을 피처 수 하나로 단정할 수는 없습니다.
>
> 여기서 챙길 원칙 하나. **교차 검증에서 선택되는 하이퍼파라미터는 피처 구성이 바뀌면 함께 바뀝니다.** 피처를 바꾼 뒤에는 이전 설정을 그대로 고정하지 않고 다시 검증합니다. 따라서 튜닝은 피처 검토 뒤에 수행합니다.
>
> 이번 탐색의 상위 두 조합에서 `min_frequency`에 따른 평균 차이는 0.0004입니다. **전처리 손잡이도 격자에 넣어 봤기 때문에** "영향이 작다"고 말할 수 있는 것이지, 이 차이의 안정성은 반복 검증이나 외부 검증으로 추가 확인해야 합니다.

# 문제 5. 실험 기록표 · 모델 저장 · 모델 카드 v5

오늘 여러 실험을 했습니다. 그 기록을 남기고, 최종 모델을 파일로 저장합니다. 기록과 학습된 `Pipeline`을 함께 남기면 실험 조건과 예측 절차를 재현하기 쉬워집니다.

```
[문제 5]
1) 지금까지 모은 log를 DataFrame으로 만들어 실험 기록표를 출력하고 CSV로 저장합니다.
   → experiment_log_v5.csv  (열: 조건 → 지표 → 결정)
2) GridSearchCV의 최적 Pipeline을 joblib으로 저장합니다.
   → purchase_pipeline.joblib   (compress 옵션을 지정합니다)
3) 저장한 파일을 다시 불러와 예측이 되는지 확인합니다.
4) 아래 모델 카드 v5 템플릿을 채웁니다.
```

> ⚠️ **주의하기 — 선택 점수와 최종 평가**  
> `best_score_`는 같은 데이터에서 후보를 비교해 얻은 내부 CV 점수이므로 낙관적으로 편향될 수 있습니다. 최종 일반화 성능은 별도 테스트셋이나 바깥쪽 교차 검증(Nested CV)으로 평가합니다.

> ⚠️ **주의하기 — 저장 파일을 다루는 규칙**  
> 저장한 `.joblib`은 파일을 여는 것만으로 그 안에 담긴 코드가 실행될 수 있습니다. **출처를 신뢰할 수 없는 파일은 열지 않습니다.** 또한 저장한 환경과 여는 환경의 `scikit-learn` 버전이 다르면 로드에 실패할 수 있으므로, 모델 파일을 남길 때 버전도 함께 기록합니다.

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [7]:
# [C6] 문제 5. 실험 기록표 · 모델 저장 · 모델 카드 v5
# ⌨️ 문제 5 — 기록표 CSV + Pipeline joblib 저장·재로드
import joblib

# 변환 후 피처 개수 (최종 파이프라인의 전처리 결과 확인)
n_features = final_pipe.named_steps["pre"].transform(feat[num_cols_p2 + CAT_COLS]).shape[1]

# 1) 실험 기록표
log = [
    {"실험": "문제1-A", "조건": "수치형10 + 범주형7 (baseline, min_freq=20, leaf=20)",
     "F1": full_scores["test_f1"].mean(), "AP": full_scores["test_ap"].mean(),
     "기준_대비_ΔF1": None, "기준_대비_ΔAP": None,
     "결정": "채택", "사유": "이후 모든 실험의 기준선으로 사용"},
    {"실험": "문제1-B", "조건": "수치형10만",
     "F1": num_scores["test_f1"].mean(), "AP": num_scores["test_ap"].mean(),
     "기준_대비_ΔF1": num_scores["test_f1"].mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": num_scores["test_ap"].mean() - full_scores["test_ap"].mean(),
     "결정": "기각", "사유": "AP가 기준보다 낮음 — 범주형 유지"},
    {"실험": "문제2-A", "조건": "기준 + has_page_value",
     "F1": scores_A["test_f1"].mean(), "AP": scores_A["test_ap"].mean(),
     "기준_대비_ΔF1": scores_A["test_f1"].mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": scores_A["test_ap"].mean() - full_scores["test_ap"].mean(),
     "결정": "채택", "사유": "F1·AP 모두 뚜렷하게 개선, EDA 근거와 일치"},
    {"실험": "문제2-B", "조건": "기준 + total_pages",
     "F1": scores_B["test_f1"].mean(), "AP": scores_B["test_ap"].mean(),
     "기준_대비_ΔF1": scores_B["test_f1"].mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": scores_B["test_ap"].mean() - full_scores["test_ap"].mean(),
     "결정": "기각", "사유": "개선폭이 폴드 변동폭 이내, 기존 피처와 중복"},
    {"실험": "문제3-누수", "조건": "기준+has_page_value + combo 타깃인코딩(전체데이터, 누수)",
     "F1": leak_scores["test_f1"].mean(), "AP": leak_scores["test_ap"].mean(),
     "기준_대비_ΔF1": leak_scores["test_f1"].mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": leak_scores["test_ap"].mean() - full_scores["test_ap"].mean(),
     "결정": "기각(사용 금지)", "사유": "검증 폴드 정답이 피처 계산에 유입된 누수"},
    {"실험": "문제3-교차적합", "조건": "기준+has_page_value + combo 타깃인코딩(TargetEncoder)",
     "F1": correct_scores["test_f1"].mean(), "AP": correct_scores["test_ap"].mean(),
     "기준_대비_ΔF1": correct_scores["test_f1"].mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": correct_scores["test_ap"].mean() - full_scores["test_ap"].mean(),
     "결정": "기각", "사유": "누수 제거 후에는 기준 대비 실질 개선 없음"},
    {"실험": "문제4-GridSearchCV", "조건": f"기준+has_page_value, {gs.best_params_} (n_estimators=300 최종)",
     "F1": f1_scores.mean(), "AP": gs.best_score_,
     "기준_대비_ΔF1": f1_scores.mean() - full_scores["test_f1"].mean(),
     "기준_대비_ΔAP": gs.best_score_ - full_scores["test_ap"].mean(),
     "결정": "채택 (최종 모델)", "사유": "전처리·모델 하이퍼파라미터 동시 탐색 결과 AP 최고 조합"},
]

log_df = pd.DataFrame(log)
print(log_df.to_string(index=False))
log_df.to_csv("experiment_log_v5.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: experiment_log_v5.csv")

# 2) 최종 Pipeline 저장
# 주의: refit=False로 탐색했으므로 gs.best_estimator_가 아니라
#      n_estimators=300으로 재학습한 final_pipe를 저장합니다.
joblib.dump(final_pipe, "purchase_pipeline.joblib", compress=3)
print("저장 완료: purchase_pipeline.joblib")

# 3) 재로드 및 예측 확인
loaded_pipe = joblib.load("purchase_pipeline.joblib")
sample = feat[num_cols_p2 + CAT_COLS].iloc[:5]
preds = loaded_pipe.predict(sample)
probs = loaded_pipe.predict_proba(sample)[:, 1]
print("\n재로드 모델 예측 확인 (샘플 5건)")
print(pd.DataFrame({"예측": preds, "구매확률": probs}))

              실험                                                                                                   조건       F1       AP  기준_대비_ΔF1  기준_대비_ΔAP         결정                               사유
           문제1-A                                                        수치형10 + 범주형7 (baseline, min_freq=20, leaf=20) 0.576372 0.737126        NaN        NaN         채택               이후 모든 실험의 기준선으로 사용
           문제1-B                                                                                               수치형10만 0.618622 0.724765   0.042250  -0.012361         기각             AP가 기준보다 낮음 — 범주형 유지
           문제2-A                                                                                  기준 + has_page_value 0.646495 0.749802   0.070123   0.012676         채택     F1·AP 모두 뚜렷하게 개선, EDA 근거와 일치
           문제2-B                                                                                     기준 + total_pages 0.579692 0.739853   0.003320   0.002727         기각        개선폭이 폴드 변동폭 이내, 

<details>
<summary>(클릭) 💡 힌트</summary>

- `pd.DataFrame(log)`로 바로 표가 됩니다. `결정` 열은 손으로 채워 넣습니다(채택/기각과 한 줄 이유).
- `joblib.dump(gs.best_estimator_, "purchase_pipeline.joblib", compress=3)` — 압축을 주지 않으면 20MB가 넘습니다.
- **`Pipeline` 통째로** 저장해야 합니다. 모델만 저장하면 전처리를 다시 만들어야 하고, 그때 설정이 어긋나면 그것도 사고입니다.
- 재로드는 `joblib.load(...)`이고, 예측에 넣는 데이터는 **학습할 때와 같은 열 구성**이어야 합니다.

</details>

## 모델 카드 v5 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330세션, 원본 예측 피처 17개 검토)
- 검증 방식: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 주 지표: AP(`average_precision`) · F1 (정확도는 참고)
- **전처리: ColumnTransformer — 수치 { }개 passthrough / 범주 { }개 One-Hot(min_frequency={ })**
  - **변환 후 피처 { }개**
- **파생변수(가설 → 결과)**
  - 가설 A `has_page_value`: { 가설 } → Δf1 { } · ΔAP { } → **{ 채택/기각 }**
  - 가설 B `total_pages`: { 가설 } → Δf1 { } · ΔAP { } → **{ 채택/기각 }**
- **누수 점검: { 만들어 본 누수와 부풀림 폭 } → 교차 적합 버전은 기준 대비 { }**
  - **학습되는 전처리의 `fit`이 `Pipeline` 안에서 일어나는가: { 예/아니오 }**
- **튜닝: GridSearchCV(scoring="average_precision"), 격자 { } → best { }**
- 내부 CV 추정: AP { } ± { } / F1 { } ± { }
- 최종 평가: { 별도 테스트셋 또는 Nested CV 결과 / 아직 미실시 }
- **산출물: purchase_pipeline.joblib (Pipeline 통째, compress=3) · experiment_log_v5.csv**
- 한계 & 다음 단계: { 예 — `PageValues`의 예측 시점 가용성 확인, 외부 검증, 피처 해석 }
- AI 사용 내역: { 무엇을 물었나 / fit 위치를 어떻게 검증했나 }

**제출:** 노트북 · `experiment_log_v5.csv` · `purchase_pipeline.joblib`을 개인 공개 저장소 main에 커밋·푸시하고, 저장소·커밋 링크를 제출합니다.

**스스로 점검하는 기준**

| 축        | 기준                                                                |
| --------- | ------------------------------------------------------------------- |
| 가설 우선 | 코드보다 가설을 먼저 적었는가                                       |
| 실험 격리 | 파생변수를 하나씩 따로 재서 원인을 가렸는가                         |
| 누수 규율 | 학습되는 전처리의 `fit`이 `Pipeline` 안에 있는가, 그것을 명시했는가 |
| 재현성    | 저장한 파일만으로 남이 같은 예측을 얻을 수 있는가                   |

> 🚀 **직접 확장하기**  
> 문제 1에서 범주형을 넣자 F1이 떨어졌습니다. 검증 데이터에서 **후보 임계값을 다시 선택**하고 별도 테스트셋에서 확인합니다. 그러면 "지표가 떨어졌다"와 "모델이 나빠졌다"가 다른 말이라는 것을 숫자로 확인할 수 있습니다.

오늘 여러분은 사용하지 않았던 범주형 피처를 연결했고, 가설 두 개 중 하나를 정직하게 기각했으며, **누수 버전의 개선 +0.011이 교차 적합 평가에서 재현되지 않음을 확인했습니다.**

성능 개선과 누수 감사는 같은 실험 기록 안에서 함께 수행해야 합니다. 그래야 점수의 계산 조건과 채택 근거를 추적할 수 있습니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>